# GNN-Guided SA for PCB Placement

GPU-accelerated training for learned PCB component placement.
Clone the repo, run experiments on all boards, generate plots.

**Paper:** [adekoya2026_gat_pcb_placement.pdf](https://github.com/elcruzo/learned-pcb-placement)

In [ ]:
!git clone https://github.com/elcruzo/learned-pcb-placement.git
%cd learned-pcb-placement
!pip install torch numpy matplotlib -q

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Parse all PCB files and show stats
import os, sys
sys.path.insert(0, '.')
from placer import parse_kicad_pcb

for f in sorted(os.listdir('data')):
    if not f.endswith('.kicad_pcb'): continue
    board = parse_kicad_pcb(f'data/{f}')
    print(f'{f:35s}  components={len(board.components):4d}  nets={len(board.nets):4d}  board={board.width:.1f}x{board.height:.1f}mm')

In [ ]:
# Run experiments on all boards
import json, time
import numpy as np
from learn import run_experiment

data_dir = 'data'
os.makedirs('results', exist_ok=True)

for f in sorted(os.listdir(data_dir)):
    if not f.endswith('.kicad_pcb'): continue
    name = f.replace('.kicad_pcb', '')
    result_path = f'results/{name}.json'
    if os.path.exists(result_path):
        print(f'SKIP {name} (already done)')
        continue
    board = parse_kicad_pcb(f'{data_dir}/{f}')
    if len(board.components) < 5:
        continue
    print(f'\n===== {name} ({len(board.components)} components) =====')
    results = run_experiment(board, name)
    with open(result_path, 'w') as fp:
        json.dump(results, fp, indent=2, default=lambda x: float(x) if isinstance(x, np.floating) else x)
    print(f'saved {result_path}')

In [ ]:
# Generate all plots
!python3 graphs.py

In [ ]:
# Display results summary
import json, os
print(f'{"Board":<25} {"Comp":>5} {"Nets":>5} {"SA Cost":>10} {"GNN Cost":>10} {"SA HPWL":>10} {"GNN HPWL":>10} {"HPWL Δ":>8}')
print('-' * 90)
for f in sorted(os.listdir('results')):
    if not f.endswith('.json') or f == 'baseline.json': continue
    with open(f'results/{f}') as fp:
        r = json.load(fp)
    if 'sa_gnn' not in r: continue
    delta = (r['sa_gnn']['hpwl'] - r['sa_baseline']['hpwl']) / r['sa_baseline']['hpwl'] * 100
    print(f'{r["pcb"]:<25} {r["components"]:>5} {r["nets"]:>5} {r["sa_baseline"]["cost"]:>10.1f} {r["sa_gnn"]["cost"]:>10.1f} {r["sa_baseline"]["hpwl"]:>10.1f} {r["sa_gnn"]["hpwl"]:>10.1f} {delta:>7.1f}%')

In [ ]:
# Display plots
from IPython.display import Image, display
import os
for f in sorted(os.listdir('results')):
    if f.endswith('.png'):
        print(f'\n--- {f} ---')
        display(Image(f'results/{f}'))